# Remember to save your work!!

Colab does not automatically save your work, please make sure to save your notebook to your google drive in the "file" menu, and open the one you were working on before through google drive instead of opening the source notebook from the github.

You can also create a fork of the originally repository and save your work there, if you know how to use github.

![save visual](https://github.com/BNL-Fermilab-RENEW/tutorials/blob/main/resources/gdrive_save.png?raw=true)

In [ ]:
from typing import Iterable, Any

import matplotlib.pyplot as plt
import numpy as np
import torch

# Regression and Classification

In this tutorial, we're going to go over the fundamentals of regression and classification, which are the two most common types of supervised learning. We will also discuss some of the best practices in machine learning, such as a train-validation-testing split. In regression problems, the objective is to learn a _continuous_ output. In classification problems, the objective is to learn a _discrete_ output. Here are some examples:
- Predicting the cost of a house from its properties, such as square footage, number of bathrooms, etc. is a _regression_ problem.
- Whether or not an image is of a cat or dog is a _classification_ problem.
- Predicting the type of animal in an image is a _classification_ problem.

## Outline 

In this course we'll go through the following exercises

- Code gradient descent for a 2D linear model
- Use pytorch to fit a parameterized model with 3 variables. 
- Fit a model that matches a sine function. 
- Write a logistic regression model 
- Learn about over and under-parameterization 

## Using an agent
If you choose to use an agent to help you, please consider the following: 
1. Try it yourself first! Think about how to approach the problem and if you are missing syntax for how to write what you are trying to express, ask a classmate first. 
2. Use a provided prompt (marked with 🤖) for each problem, and try to describe what you are struggling with. Paste your current code or point the in-notebook agent to your current work.
3. Test the result and reflect! What does the code do, and how are other ways you could have approached the solution. What does the resulting code do? 

Please remember the agent shouldn't be considered a homework machine here. 
If you skip trying to understsand step 1, you won't learn anything from step 2!
These tutorials are designed to walk you how to think about these sort of problems, not give you a infinitely repeatable recipe of how to do machine learning.

## Other Resources

- [Andrew Ng's flagship Coursera course on machine learning](https://www.coursera.org/specializations/machine-learning-introduction)
- [15 types of regression](https://www.listendata.com/2018/03/regression-analysis.html#Linear-Regression)
- [Linear regression gradient descent tutorial](https://towardsdatascience.com/linear-regression-using-gradient-descent-97a6c8700931)
- [Logistic regression tutorial](https://towardsdatascience.com/logistic-regression-detailed-overview-46c4da4303bc)
- [Google's excellent Decision Forest Developers](https://developers.google.com/machine-learning/decision-forests/decision-trees)

# Regression modeling

To model a regression problem, there are 5 main pieces. 
We'll frame this with a specific problem.


1. Data Features. 
2. Data Labels. Also called "targets". This is the thing we want the model to replicate. 
3. Model & Model Parameters. Something that takes the data features and outputs the data labels. 
4. Loss Function. How we decide how good the output of the model is compared to what data we have. 
5. Optimizer. How the model/model parameters updates are calculated. 

This same recipe will apply for a lot of different ML methods, but they will look very different based on your specific problem. 

In [ ]:
# This code generates and plots the data we'll be working with
# It's very common for people to explain and test their methods with completely fake data

def generate(n_samples: int = 100, seed=42) -> tuple[np.ndarray, np.ndarray]: # the -> and : denote 'type hints'.
    # This function is equivelant to a f(x) = y. 
    def f(x):
        return 2*x[0] + 9*x[1] + 0.25 
    
    rand = np.random.default_rng(seed=seed)
    features = np.array([np.linspace(0, 10, n_samples), np.linspace(0, 10, n_samples)]) + rand.uniform(0, 2, size=(2, n_samples))
    labels = f(features) + rand.uniform(low=-0.005, high=0.005, size=n_samples)
    return features, labels

x, y = generate()

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(xs=x[0], ys=x[1], zs=y, alpha=0.8)
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.set_zlabel(r"$y$")
plt.show()

# Linear Regression

As the name implies, linear regression uses a linear function ($y=mx+b$) to fit data.
The trick is, all the parameters don't have to be 1D. 

Let's look at a 2D example.

In [ ]:
def linear_model(x, m, b, with_noise=False):
    """2D Linear data with noise"""
    y = (m[0] * x[0]) + (m[1] * x[1]) + b
    if with_noise:
        noise = np.random.default_rng(seed=42).normal(scale=4, size=x.shape[1])
        y += noise
    return y

Let's assume we don't know the actual parameters, so how do we evaluate how close we are to the solution? 

We can just evaluate the true value at a point, checking the true value ($y$) against the output of the model at point $x$ so $\hat{y} = f(x)$

There are many *many* different equations you can use to check this error, but one of the most common ones is **Mean Squared Error**

$$
L(y, \hat{y}) = \frac{\sum (y - \hat{y})^2}{|y|}
$$

We denote it with an $L$ for **loss** - the term for the difference between desired output of an ML model and the desired (true) results. 

In [ ]:
def mean_squared_error(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

## Step 1 - Take a guess at $m$ and $b$

For the above example, write your guesses for the value of $m$ and $b$, we'll use this to write a "fit". Afterwards, we can use this fit to make some predictions about future values of the line, and evaluate how good they are in comparison to our existing examples of the line! 

In [ ]:
m_guess = ["?", "?"]
b_guess = "?"

y_pred = linear_model(x, m_guess, b_guess)

In [ ]:
mse = mean_squared_error(y_true="?", y_pred="?")
mse

In [ ]:
# Let's also do a visualization!

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(x[0], x[1], y, alpha=0.5, color='grey', label="Data")
ax.plot(x[0], x[1], y_pred, color='red', label=f'Guess with MSE of {round(mse, 2)}')

plt.legend()
plt.show()

# Step 2 - Using Gradient Descent

It stands to reason that just randomly guessing parameters is probably not the most effective way of finding the right guess. Can we do better? Absolutely! [Gradient descent](https://en.wikipedia.org/wiki/Gradient_descent), and its family of related methods (you may have heard of [Adam](https://arxiv.org/abs/1412.6980)), can be used to find an optimal set of parameters for arbitrary models.

Given a function $f(x)$ and its derivative $\nabla f(x),$ the general rule of gradient descent is that to find a point closer to the actual minimum of $f,$ we follow its gradient:

$$x_{n+1} = x_n - \gamma \nabla f(x_n).$$

This function shows us how to make a step in a given space (real or digital!) to the next closest position according to our current position.

To do that we have to combine two equations - the model equation $f(x)$ and the loss $L(y, \hat{y})$

$$

L(y, \hat{y}) = \frac{\sum (y - (m*x+b))^2}{|y|}

$$

To get the best model, we just need to find the smallest value of $\nabla{L(y, \hat{y})}$

Computing that $\nabla$ can be annoying, so generally we use packages to compute those gradients for us.

### Side Note: Gradients and Derivative 

Confusingly, the words "gradients" ($\nabla f(x)$) and "derivative" ($\frac{df(x)}{dx}$) are used interchangably sometimes. For our single variable function, $f(x) = (x-2)^2$, they are the same. However, if we had a function of multiple variables (as we almost always do in ML/AI), this wouldn't be the case. For example, if we had a function $f(x,y,z) = x+y+z$, the derviative $\frac{df(x,y,z)}{dx}$ is only depending on $x$, so we wouldn't have derviative information about the other variables (y, z). We could very easily optimize a function to get to a minimum for x, but the other variables would be out in the cold. This is were the vector form of the derviative, the gradient comes in. The gradient takes the deriative of all these variables at once. 

So for our example $f(x,y,z) = x+y+z$, 

$$
\frac{df(x, y, z)}{dx} = 1+y+z 
$$ 
$$
\nabla f(x, y, z) = <\frac{df(x, y, z)}{dx}, \frac{df(x, y, z)}{dy}, \frac{df(x, y, z)}{dz}>
$$

# Use PyTorch to write the model

In python, there are many ways to write groups of parameters and automatically calculate gradients.
For this class, we'll use [PyTorch](https://docs.pytorch.org/tutorials/beginner/deep_learning_60min_blitz.html), which the standard AI/ML package used by basically all modern AI practioners.
This does require defing our model in terms PyTorch can use, so it requires us to re-write our model. 
To do automatic gradient calculations, all parts of the model have to be defined as [`Tensors`](https://docs.pytorch.org/docs/2.12/tensors.html), which have gradients. 

PyTorch models are a `child class` of [`torch.nn.Module`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.Module.html), which is how it keeps track of gradients and model parameters. 
We are going to short cut this because our model is so simple, but it'll show up in this form later. 

The models must have "parameters", with gradients attached, so they can be updated using a gradient update. 
They also must have operations done on them that "differenciable", so the gradient can be computed.
Let's redefine our linear model and loss using PyTorch to take advantage of these automatic gradient calculations.

In [ ]:
def pytorch_linear_model(x:torch.Tensor, m_parameter: torch.nn.Parameter, b_parameter: torch.nn.Parameter) -> torch.Tensor:
    "y = m*x+b"
    return torch.nn.functional.linear(x.T, m_parameter.T) + b_parameter

In [ ]:
def gradient_descent_linear_model(x:torch.Tensor, y:torch.Tensor, m0:Iterable[float], b0:float, gamma:float=0.1, N:int=100):
    """
    Preform gradient descent for a 2D linear model
    """
    lr = torch.Tensor([gamma])  # Converted to a tensor so we can do some matrix multiplication
    b = [b0]
    m = [m0]
    current_m = torch.nn.Parameter(data=torch.Tensor(m0).reshape(2, 1), requires_grad=True)  # Parameter is a part of a model, it has a accessable gradient
    current_b = torch.nn.Parameter(data=torch.Tensor([b0]).reshape(1, 1), requires_grad=True)

    for _ in range(N):
        y_pred = pytorch_linear_model(x, current_m, current_b)
        # Calculate the derivatives of the loss function at x and y, y_pred
        loss = torch.nn.functional.mse_loss(input=y_pred, target=y.unsqueeze(dim=-1))
        loss.backward()  # Using "backwards" computes the gradients

        # Update
        # Equ. to x_n+1 = x_n - ɣ ∇f(x_n)
        current_m.data = current_m.data - lr*current_m.grad
        current_b.data = current_b.data - lr*current_b.grad

        m.append([current_m.data[0].item(), current_m.data[1].item()]) 
        b.append(current_b.item())
        
        
    return m, b


In [ ]:
m, b = gradient_descent_linear_model(torch.Tensor(x), torch.Tensor(y), m0=(0.2, 6), b0=8, gamma=5e-6, N=60)

$m$ and $b$ change as a function of the number of steps (or **epochs**). 

If we plot the convergence of `m` and `b` as a function of "epoch" (or number of full passes through the training data) we can see that the data very quickly settles on a final pair of parameters. 

In [ ]:
# Progress as a function of epoch

epochs = range(len(b))

plt.axhline(m[-1][0], color="black", alpha=0.5) # Final m value, x[-1] is the last value in the list x.
plt.axhline(m[-1][1], color="black", alpha=0.5)
plt.axhline(b[-1], color="black", alpha=0.5)

plt.scatter(epochs, np.array(m)[:, 0], label="$m_1$")
plt.scatter(epochs, np.array(m)[:, 1], label="$m_2$")
plt.scatter(epochs, b, label="$b$")

plt.legend()
plt.xlabel("Epoch")
plt.title("History of $m$ and $b$ parameters")
plt.show()

We can also look at the linear fit itself at a few different steps. 

In [ ]:
# Show the different guesses over-layed onto data

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

epochs = [0, 5, 10, 20, 30, 60]
for epoch_check in epochs: 
    m_guess = [np.array(m)[epoch_check, 0], np.array(m)[epoch_check, 1]]
    b_guess = b[epoch_check]
    x_range = np.arange(0, 10)
    y_pred = x_range*m_guess[0] + x_range*m_guess[1] + b_guess

    ax.plot(x_range, x_range,  y_pred, label=f"Epoch {epoch_check}")

ax.scatter(x[0], x[1], y, label='True', alpha=1, )
ax.set_xlabel("x")
ax.set_ylabel("f(x)=mx+b")
ax.legend()
#ax.set_title(f"Progress for finding f(x)={round(m[0, -1],2)}x_1+{round(m[1, -1],2)}x_2+{round(b[-1],2)}")

plt.show()

This is pretty unwieldly way to look at our progress, so we can make something called "A Loss Curve". This gives one number for the whole model, so we have one point per epoch. 

Generally speaking, in ML we have a lot of parameters to track, so instead of storing every single model in our training process, we just keep track of this loss instead as a proxy for model performance. 

In [ ]:
loss_history = []
for m_guess, b_guess in zip(m, b): 
    y_pred = pytorch_linear_model(
        torch.tensor(x), 
        torch.Tensor(m_guess).to(torch.double).reshape(2, 1), 
        torch.Tensor([b_guess]).to(torch.double)
    )
    loss = torch.nn.functional.mse_loss(input=y_pred, target=torch.Tensor(y).unsqueeze(dim=-1))
    loss_history.append(loss)

epoch = range(len(loss_history))
plt.scatter(epoch, loss_history, alpha=0.5)

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Loss History for f(x)=mx+b") 
plt.show()

# Optimizers

This training scheme uses our custom gradient descent method, where we compute all the model parameter updates manually. 
It's pretty unusual to train a model this way, typically pytorch models use a build in [`optimizer`](https://docs.pytorch.org/docs/main/optim.html) to update parameters.

An optimizer object updates the parameters passed to it according to a specific algorithm. 
Here we'll use `stochastic gradient descent` or [`sgd`](docs.pytorch.org/docs/main/generated/torch.optim.SGD.html#torch.optim.SGD), which is a varient of gradient descent that is better suited to machine learning applications. 

In [ ]:
# Changing our update loop to use an optimizer instead of a manual param update
def gradient_descent_linear_model(x:torch.Tensor, y:torch.Tensor, m0:Iterable[float], b0:float, gamma:float=0.1, N:int=100):
    """
    Preform gradient descent for a 2D linear model
    """
    b = [b0]
    m = [m0]
    current_m = torch.nn.Parameter(data=torch.Tensor(m0).reshape(2, 1), requires_grad=True)  # Parameter is a part of a model, it has a accessable gradient
    current_b = torch.nn.Parameter(data=torch.Tensor([b0]).reshape(1, 1), requires_grad=True)
    optimizer = torch.optim.SGD(params=[current_m, current_b], lr=gamma)


    for _ in range(N):
        y_pred = pytorch_linear_model(x, current_m, current_b)
        loss = torch.nn.functional.mse_loss(input=y_pred, target=y.unsqueeze(dim=-1))
        loss.backward()  # Using "backwards" computes the gradients
        optimizer.step() # Perform the update on parameters

        m.append([current_m.data[0].item(), current_m.data[1].item()]) 
        b.append(current_b.item())
    return m, b

m, b = gradient_descent_linear_model(torch.Tensor(x), torch.Tensor(y), m0=(0.2, 6), b0=8, gamma=5e-6, N=60)

In [ ]:
loss_history = []
for m_guess, b_guess in zip(m, b): 
    y_pred = pytorch_linear_model(
        torch.tensor(x), 
        torch.Tensor(m_guess).to(torch.double).reshape(2, 1), 
        torch.Tensor([b_guess]).to(torch.double)
    )
    loss = torch.nn.functional.mse_loss(input=y_pred, target=torch.Tensor(y).unsqueeze(dim=-1))
    loss_history.append(loss)

epoch = range(len(loss_history))
plt.scatter(epoch, loss_history, alpha=0.5)

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Loss History for f(x)=mx+b") 
plt.show()
# Should show a very similar result

# Validating your model

A big part of model training is convincing yourself the model is accurate. 
This means that when you train yourself, it makes sense to make sure not all of the data you have goes into the model. 
If you don't, you are basically training your model on the anwser guide. 


To do this, we'll get a set of data from the original dataset that the model has not seen, and the model performs the same on this.

In [ ]:
x_test, y_test = generate(n_samples=50, seed=41) # Using a different seed to get different data
# We can also "split" the data by using an index to make two subsets of the data

# We'll get the last parameter we guessed
last_m = m[-1]
last_b = b[-1]
new_dataset_prediction = pytorch_linear_model(
        torch.tensor(x_test), 
        torch.Tensor(last_m).to(torch.double).reshape(2, 1), 
        torch.Tensor([last_b]).to(torch.double)
    )

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

ax.scatter(x_test[0], x_test[1], y_test.squeeze(), label='True')
ax.scatter(x_test[0], x_test[1], new_dataset_prediction.squeeze(), label='Test')

ax.set_xlabel("x")
ax.set_ylabel("f(x)=mx+b")
ax.legend()

# How to write a model-update loop

Let's put all the steps we just used together. 

1. Define your dataset
    
    This step can be cleaning a pre-existing dataset to suit your needs, gathering data, or just putting it in a form your model needs. 
    It is harder to simulate this step, so in this notebook we are generating data and transforming it into torch Tensors. 

2. Define the model with the set of parameters you want to optimize

    This is were we define how we're fitting data

3. Define a loss function

    How do we define how good a model is fit? It should take both your target data and the output of your model. 
    Mean Squared Error is a common pick for regression problems, but it can be any number metrics. 

    In some problems, this is as simple as saying "Is what the model gave correct"? But it can vary if extra considerations are needed (such as, how important is it if the model learns certain cases vers others?)

    Other methods besides MSE already avaliable in torch can be seen [here](https://docs.pytorch.org/docs/2.12/nn.html#loss-functions), but you can also code a new version yourself!

4. Define an optimization/parameter update method. 

    In torch this is done with using a `torch.optim` optimizer method, but we'll see this in action in the below example


5. Define a way to validate a result.

    This is simular, but not the same as the loss function.
    We need to test against a different dataset than that we train on. 
    While we can use the same loss function to verify the result (and it is important), we can also quanitify the result using different metrics. 

Below is a bit of template code to reference. 

In [ ]:
# Create/clean your dataset
def generate_data():
    x = torch.Tensor()
    y = torch.Tensor()
    return x, y

# Define a model
# Uses torch.nn.Module
class SampleModel(torch.nn.Module): 
    def __init__(self):
        super().__init__()

        # What will be optimized
        # Can also be different sort of layers when we talk about deep and convolutional neural networks. 
        self.param_1 = torch.nn.Parameter()

    def forward(self, x): 
        # What do we do with the parameters and the input data to get an output?
        return self.param_1 @ x

In [ ]:
def new_generate(n_samples=400):
    # Our data here is 4d parabola
    random = np.random.default_rng(seed=42)
    n_dim = 4
    low = -6
    high = 10
    
    # Generate coefficients for the 4D parabola
    # Form: f(x) = sum(a_i * x_i^2) + sum(b_i * x_i) + c
    a_coeff = random.integers(low, high, size=(1, n_dim)).astype(np.float32)
    b_coeff = random.integers(low, high, size=(1, n_dim)).astype(np.float32)
    c_coeff = random.integers(low, high)
    
    def f(x):
        # Element-wise squaring for the quadratic terms
        quadratic_term = np.sum(a_coeff * x**2, axis=1, keepdims=True)
        linear_term = np.sum(b_coeff * x, axis=1, keepdims=True)
        return quadratic_term + linear_term + c_coeff

    noise = random.uniform(low=-1, high=1, size=(n_samples, n_dim)).astype(np.float32)
    x = np.linspace((-10,-10, -10, -10),(20, 20, 20, 20), n_samples) + noise
    y = f(x)
    return torch.tensor(x).to(torch.float), torch.tensor(y).to(torch.float), (a_coeff, b_coeff, c_coeff)

In [ ]:
x, y, true_params = new_generate()

# Plot them to get a vibe here
# X has multiple dimensions here, so let's make the same number of plots

fig, ax = plt.subplots(2, 2)
ax[0, 0].scatter(x[:, 0], y)
ax[0, 1].scatter(x[:, 1], y)
ax[1, 0].scatter(x[:, 2], y)
ax[1, 1].scatter(x[:, 3], y)
fig.tight_layout()
fig.suptitle("")
plt.show()

In [ ]:
class Model(torch.nn.Module):
    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)

        # Include the model parameters here
        # torch.nn.module collects these into something the gradient optimizer can pick up
        
        # Our fake model here matches the form of the data, but not the coeffients
        # Because we don't know what they are! 
        # We'll initialize with Ones.
        self.a = torch.nn.Parameter(data=torch.zeros(1, 4).to(torch.float))
        self.b = torch.nn.Parameter(data=torch.zeros(1, 4).to(torch.float))
        self.c = torch.nn.Parameter(data=torch.zeros(1).to(torch.float))

    def forward(self, x): 
        # f(x) = sum(a_i * x_i^2) + sum(b_i * x_i) + c
        quadratic_term = torch.sum(self.a * x**2, dim=1, keepdim=True)
        linear_term = torch.nn.functional.linear(x, self.b)
        return quadratic_term + linear_term + self.c

In [ ]:
def validation_step(x, y, model):
    model.train(False)
    with torch.no_grad():
        y_pred = model(x)
        loss = torch.nn.functional.mse_loss(y_pred.squeeze(), y.squeeze())
    return loss.item()

def training_step(x, y, model, optimizer): 
    model.train(True)
    optimizer.zero_grad()
    y_pred = model(x)
    loss = torch.nn.functional.mse_loss(y_pred.squeeze(), y.squeeze())
    loss.backward()
    
    # Clip gradients to prevent explosion
    # This isn't always necessary, but this problem has so few parameters 
    # It's a little strange for a pytorch construction. 
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)    
    optimizer.step()

    return loss.item()


def train(x, y, epochs, lr):
    model = Model()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    # Use the first 75% of the dataset for training, the last 25% for validation
    break_index = int(.75 * x.shape[0])
    x_train, x_valid = x[:break_index], x[break_index:]
    y_train, y_valid = y[:break_index], y[break_index:]

    loss_history = []
    loss_history_validation = []
    for _ in range(epochs):
        training_loss = training_step(x_train, y_train, model, optimizer)
        validation_loss = validation_step(x_valid, y_valid, model)

        loss_history.append(training_loss)
        loss_history_validation.append(validation_loss)

    return model, loss_history, loss_history_validation

trained_model, loss_history, loss_history_validation = train(x, y, epochs=50, lr=1e-1)


In [ ]:
epochs = range(len(loss_history))
plt.plot(epochs, loss_history)
plt.plot(epochs, loss_history_validation)

In [ ]:
# And we can check this model on a new set of data and inspect the actual final parameters
test_samples = torch.tensor(np.linspace((-10,-10, -10, -10),(20, 20, 20, 20), num=100)).to(torch.float)

a = true_params[0][0]
b = true_params[1][0]
c = true_params[2]

actual_func = f"{a[0]}x**2 + {a[1]}x**2 +{a[2]}x**2 + {a[3]}x**2 + " \
    f"{b[0]}x + {b[1]}x + {b[2]}x + {b[3]}x + " \
    f"{c}"

model_out = trained_model(test_samples).detach()
true_out = torch.sum(torch.tensor(a).T * test_samples**2, dim=-1, keepdim=True) \
      + torch.nn.functional.linear(test_samples, torch.tensor(b).T).unsqueeze(dim=-1) + c

fig, ax = plt.subplots(2, 2)

for result, label, color in zip([model_out, true_out], ["Prediction", "True"], ["blue", "red"]):
    ax[0, 0].scatter(test_samples[:, 0], result, label=label, color=color, alpha=0.6)
    ax[0, 1].scatter(test_samples[:, 1], result, label=label, color=color, alpha=0.6)
    ax[1, 0].scatter(test_samples[:, 2], result, label=label, color=color, alpha=0.6)
    ax[1, 1].scatter(test_samples[:, 3], result, label=label, color=color, alpha=0.6)

ax[0,1].legend()
fig.suptitle(f"Model vs True for \n {actual_func}")
plt.show()


In [ ]:
# Extract the actual fit

a = np.array(trained_model.a.data)[0]
b = np.array(trained_model.b.data)[0]
c = np.array(trained_model.c.data)[0]


trained_func = f"{a[0]}x**2 + {a[1]}x**2 +{a[2]}x**2 + {a[3]}x**2 + " \
    f"{b[0]}x + {b[1]}x + {b[2]}x + {b[3]}x + " \
    f"{c}"

print(trained_func)
# This isn't Exactly what the data is, but the fit is the same! This is the problem with these higher level fits, 
# There are multiple local minimum
# The loss stablized but it isn't the same solution!

# Bonus Problem - Expand the range of model 

Instead of just using this -10, 20 range, try to example the data range by modifying the `test_range` variable. 

How well does the model hold up to the new data range? How do you correct the training proceedure to fix it?


In [ ]:
new_test_samples = torch.tensor(np.linspace((-30,-30, -30, -30),(-10, -10, -10, -10), num=100)).to(torch.float)


original_trained_model_out = trained_model(new_test_samples).detach()
true_out = torch.sum(torch.tensor(a).T * new_test_samples**2, dim=-1, keepdim=True) \
      + torch.nn.functional.linear(new_test_samples, torch.tensor(b).T).unsqueeze(dim=-1) + c

fig, ax = plt.subplots(2, 2)

for result, label, color in zip([model_out, true_out], ["Prediction", "True"], ["blue", "red"]):
    ax[0, 0].scatter(new_test_samples[:, 0], result, label=label, color=color, alpha=0.6)
    ax[0, 1].scatter(new_test_samples[:, 1], result, label=label, color=color, alpha=0.6)
    ax[1, 0].scatter(new_test_samples[:, 2], result, label=label, color=color, alpha=0.6)
    ax[1, 1].scatter(new_test_samples[:, 3], result, label=label, color=color, alpha=0.6)

ax[0,1].legend()
plt.show()

# How should we improve the model?

# Improving generalization

'Generalization' refers to getting a model to fit data outside the exact points included in the model training.
This can be paired with the concept of 'overfitting', where a model only learned the data as it was presented. 
This can cause a lot of problems when the model comes in contact with the real world, because it encounters data outside of what it was tuned to deal with.

There a few methods that can be used to improve the generalization of a result. 

* Increase the volume and variety of the data used to train the model - More data results in more examples to learn
* Increase the number of parameters used - More parameters are harder to learn, and the min. of the loss gradient is less likely to fall into a local mininium. 
* Change the optimization function - Using an optimizer with an adaptive LR or other techniques can help the loss escape local minimum.
* Add noise to the training data. The data does not perfectly correlate to the output, so the model must account for this. 
* Normalize the input data. 

🤖 I am writing a pytorch model that can fit a 4D polynominal ranging from (-100, 100). I am not using a deep neural network. Include a training scheme that can handle out-of-domain testing data. The input data should only cover (-50, 0), (25, 75). The test data covers the rest of the range. The true polynominal is of degree 2 and has the coeffs. stored in the variables a, b, c. Assume those variables already exist. Help me write an outline of the steps to take to complete this model. 

In [ ]:
class AdvancedModel(torch.nn.Module):
    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)

        ...

    def forward(self, x): 
        # f(x) = sum(a_i * x_i^2) + sum(b_i * x_i) + c
        ...

In [ ]:
def new_validation_step(x, y, model):
    model.train(False)
    with torch.no_grad():
        ...


def new_training_step(x, y, model, optimizer): 
    model.train(True)
    optimizer.zero_grad()
    ...

def train(x, y, epochs, lr):
    model = AdvancedModel()
    optimizer = ...

    # Use the first 75% of the dataset for training, the last 25% for validation
    break_index = int(.75 * x.shape[0])
    x_train, x_valid = x[:break_index], x[break_index:]
    y_train, y_valid = y[:break_index], y[break_index:]

    loss_history = []
    loss_history_validation = []
    for _ in range(epochs):
        training_loss = training_step(x_train, y_train, model, optimizer)
        validation_loss = validation_step(x_valid, y_valid, model)

        loss_history.append(training_loss)
        loss_history_validation.append(validation_loss)

    return model, loss_history, loss_history_validation

trained_model, loss_history, loss_history_validation = train(x, y, epochs=50, lr=1e-1)

# Group Exercise - Fit an arbitrary Function

Instead of just finding a minimum of a single point function, we can also find a match for a function. 
This is useful for modeling behavior of a dataset, like if you have a dataset and want know the value of an input not contained directly in the data. 

For the example below, lets pretend we are trying to find out the tone of a sound wave we measured at a couple of points. 

For this, we will use the base equation $ y(t) = A sin(2\pi(ft + \phi)) $, so the variables to fit are $A, f, \phi $ 

## Steps: 

1. Write a function (loss) that will evaluate how close your guess of parameters is to the data
2. Write a function that will iteratively update your parameter guess 
    * Pick input parameters for the function that include how many epochs, learning rate, and initial value guess
    * Use the defined gradient functions to calculate the update of each variable
    * Return the final variables and the loss at each step
3. Plot the loss against the number of guesses made
4. Plot the final function against the input data

🤖 I am using pytorch to write an nn.Module model that has training parameters that can be fit to a 1D sine function. I am not use deep neural networks or other techniques, just including parameters that directly map to the shape of a given sine curve. Include a training loop with training and validation data and a way to plot the finished result. Assume the source dataset already exists via the function "wave_data" which takes the parameter "N_points". Write a plan for me to complete this task. 

In [ ]:
def wave_data(N_points, noise=0.1):
    """
    Generate data for a sine wave with added noise
    """
    x = np.random.default_rng().random(N_points)*5
    base_points = 2 * np.sin(3.2 * x + 1.4)
    y = base_points + noise * np.random.randn(N_points)
    return torch.tensor(x), torch.tensor(y)


In [ ]:
x, y = wave_data(100)

plt.scatter(x, y, alpha=0.5, color='grey', s=10)
plt.xlabel("Time $t$")
plt.ylabel("Pitch $y$")
plt.title("Collected Pitch Data")

In [ ]:

class WaveModel(torch.nn.Module):
    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)

        # What parameters do we need to express a sine wave? 
        self.A = torch.nn.Parameter(...)

    def forward(x):
        # How do we write y = A*sin(B*x + C)? 
        pass


def loss_function(y_pred, y_true):
    """
    Write a loss function to be used for fitting your function to the data
    """
    loss = "??"
    return loss

def optimize_loss(x, y, n_epochs):
    """Write a function that optimizes the loss function, and returns the best fit parameters"""
    
    a = "??"
    f = "??"
    phi = "??"


    loss = []

    # a, f, and phi will be tensors because you use a tensor to calculate the gradient
    # We rectify this by taking only the numpy value (numeric, no gradient attached) at the end
    return trained_model, loss

In [ ]:
def plot_loss(loss): 
    "Plot how the loss changes over time"
    epochs = range(len(loss))
    plt.plot(epochs, loss)
    plt.xlabel("Epoch")
    plt.ylabel("Loss Value")
    plt.title("Loss History")
    plt.show()


def plot_fit(trained_model, x_test, y_true): 
    "Plot the best fit relative to the true values"

    y_pred = trained_model(x_test)
    
    f = "?"
    a  = "?"
    phi = "?"

    plt.scatter(x, y_true, alpha=0.5, color='grey', s=10, label='True Data')
    plt.scatter(x, y_pred, color='red', label='Fitted Data')

    plt.title(f"Fit with y = {round(a, 3)}sin(2π({round(f, 3)}x + {round(phi, 3)}))")
    plt.show()

In [ ]:
trained_model, loss = optimize_loss(x, y, n_epochs=1000)
plot_loss(loss)
plot_fit(trained_model, x, y)

# Classification

Until now, we have assumed that when you give an input, you get an output number. 
This is very useful, but not always what we want. 
Let's imagine you accidentally erased all your contacts and are now getting cling texts from your ex.
You could have a number of parameters about the text - how many characters it is, when it was sent, how many times they had texted before, etc. 

You want to make sure you never get texts from them, but texts from every else. 
In this case, you want to put the texts you get into 2 groups - ones you want to read and ones you don't. 

This is a `classification` problem - putting the input data into defined groups. 

There are many sort of algorithms to deal with this sort of data, but we'll start with the one most like `linear regression`. 

# Logistic Regression 

Logistic regression is a slight modification of linear regression that allows for categorical targets. There are multiple types of logistic regression, but for this tutorial, we will focus on _binary_ logistic regression: when the target value can be "on or off": 0 or 1.

Ignore that the name calls it a "regression" technique, this is classification. 

Consider the previous linear regression model of a single variable: $f(x) = mx + b$. The range, or set of possible values, that $f(x)$ can take on is infinite: $f(x) \in (-\infty, \infty).$ This is appropriate, in general, for regression problems in which the output can take on any value, but in binary classification problems, we want the model to essentially represent the probability of being one of the two classes.

So, for logistic regression, we add an extra step:

$$g(x) = \sigma(f(x))$$

where 

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

is called the _sigmoid function_. The sigmoid function has the property of being bounded between 0 and 1, making it a convenient choice for our purposes.

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

In [ ]:
grid = np.linspace(-6, 6, 100)
plt.plot(grid, sigmoid(grid))
plt.xlabel("$x$")
plt.ylabel("$\sigma(x)$")
plt.title("$\sigma(x) = 1/{1 + e^{-x}}$")
plt.show()

#  Assigning Loss and making the model

For this model, we are going to use a new loss function. 
This function, called **cross-entropy loss** counts the number of correct classifications (0 or 1) vs the incorrect classifications. 

$$J(\hat{y}, y) = -y \ln \hat{y} - (1-y) \ln (1-\hat{y}),$$

and the total loss over the dataset:

$$L(g(\vec{x}), y) = \frac{1}{N} \sum_{i=1}^N J(g(x_i), y_i).$$

Don't worry about perfectly writing this, we will be using torch to compute this loss again.

In [ ]:
# Import SKLearn modules
from sklearn.datasets import make_moons # Making a dataset!
from sklearn.model_selection import train_test_split # Shortcut for the "Split the data"

In [ ]:
def generate_classification_data(N_points):
    """
    Generate data for a 2D binary classification problem
    """
    x, y = make_moons(N_points, noise=0.1)
    return x, y

x, y = generate_classification_data(1000)

plt.scatter(x[:, 0], x[:, 1], c=y, cmap=plt.cm.RdYlBu, s=10)
plt.xlabel("$x_1$") 
plt.ylabel("$x_2$")
plt.title("Generated Data")
plt.show()

## Using the model 

You may look at the sigmoid function and the data and think. 
"Wow there's no way you can map that sigmoid point to this data!" 
And you'd be correct. 
This is an example of non-linear data being fit with a linear function, which is sub-optimal. 
It is always a good idea to pick a model that has the appropriate amount of expressiveness to capture the data.
But, for the sake of this exercise, let's see how well we do otherwise. 

Before we start training, we will be pulling aside some data to test the results of the training. 
If we did not do this, it is possible that the data we test on would be directly used to optimize the model, letting it cheat.

🤖 Help me write a training scheme usubg an existing LogisticRegression pytorch model to fit a binary classification problem with data stored in 'x_train, x_test, y_train, y_test' variables. Assume this data already exists. Help me outline how to use this model to write a training loop that validating my results with the test dataset.

In [ ]:
class LogisticRegression(torch.nn.Module):    
    def __init__(self, n_inputs):
        super(LogisticRegression, self).__init__()
        # torch.nn.Linear is equ. to a = torch.nn.parameter(...); y = torch.nn.functional.linear(a, x)
        self.linear = torch.nn.Linear(n_inputs, 1) # One output, result of 0, 1

    def forward(self, x):
        y_pred = torch.sigmoid(self.linear(x)) # Apply the sigmoid to the linear output
        return y_pred

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss() # Cross entropy loss given by PyTorch

In [ ]:
# Split the data into training and test sets

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

model = LogisticRegression(n_inputs=x_train.shape[-1]) # How many dimensions is the data? 


# How do we train this model?
# It isn't any different from the original proceedure
n_epochs = 20
for epoch in range(n_epochs):
    # train
    ...

    # validate
    ...


# Get the result from the model 
y_pred = model(x_test)

fig, ax = plt.subplots(1, 2, figsize=(15, 5))

ax[0].scatter(x_test[:, 0], x_test[:, 1], c=y_test, cmap=plt.cm.RdYlBu, s=10)
ax[0].set_title("Test Data")

ax[1].scatter(x_test[:, 0], x_test[:, 1], c=y_pred, cmap=plt.cm.RdYlBu, s=10)
ax[1].set_title("Predicted Data")

fig.supxlabel("$x_1$")
fig.supylabel("$x_2$")
fig.suptitle(f"Testing Data with an accuracy of {model.score(x_test, y_test)}")

plt.show()

## Polynominal Fits, Underfitting, and Overfitting

This sample principals for optimization can be applied to anything you can take a derivative of. (This includes complicated neural network architectures! Basically all ML is just increasing complicated linear algebra!)

So instead of fitting a simple line of data, let's try a more complicated data structure, so we can optimize a polynominal to fit. 

In [ ]:
from numpy import poly


rng = np.random.default_rng(seed=42)

n_coeff = rng.integers(4,9) # Anywhere between 4 and 9 polynominal coefficents
poly_coefficents = [rng.random() for _ in range(n_coeff)]
polynominal = poly.polynomial.Polynomial(poly_coefficents) # Random coefficents, we don't know what the polynominal we're fitting is (unless we look ;))
x, y = polynominal.linspace(n=100, domain=[-5, 5])

y += rng.uniform(-3, 3, 100)

plt.scatter(x,y)

### Guess a number of parameters, and optimize

`Np.Polynominal` has a least square fit included, so we can use this a shortcut instead of writing a whole new gradient descent method. Though, remember, as long as you _can_ take a derivative, you can try to optimize this model via gradient descent! 

In [ ]:
# coefficent guess

# This looks almost like a parabola, so let's try a 2nd degree polynominal for a fit 
degree_of_fit = 2
new_model = poly.polynomial.Polynomial([1 for _ in range(degree_of_fit+1)]) # You can play around with these to try and get a good initial fit! 
model_fit = new_model.fit(x, y, deg=2)
x_fit, y_fit = model_fit.linspace(n=100, domain=[-5, 5]) # Get points over the same space, see the loss! 

mean_squared_error(y, y_fit)

In [ ]:
plt.scatter(x, y, alpha=0.4, label='True')
plt.scatter(x_fit, y_fit, label='Fit')

plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.title(f"Fit y={model_fit}")
plt.show()

This isn't a bad fit, strictly speaking, but it's not great. We can probably call this model `underfit` (as we have more complexity in the data than the model predicts.) Look at how the tail of the model, (around x=-4) behaves as compared to the data - there is a mis-match. 

Model underfitting can be caused by many things but the two that are easiest to correct are:

* Model does not have enough parameters to fit the data 

If your model is too simple for your data, it'll be hard to show good results. Imagine trying to draw a line of best fit through a sine function - you'll never get perfectly accurate results. 

* Model hasn't been trained long enough. 

This is like if we stopped training the linear model at epoch 10. There is still a lot of extra work the model can do, it hasn't converged on a solution. This can be tricker to see when you get more complicated data and more complicated models, but keeping a second set (called a `validation` set) which is only used for testing can help. When the training data doesn't improve beyond the validation data, that indicates there may be a problem. 


In the above example, we know for a fact the model is unfit with too few parameters, as we tried to fit a polynominal that is at least degree 4 with a degree 2 model. Let's try again with a more complicated model. 

In [ ]:
degree_of_fit = 16
new_model = poly.polynomial.Polynomial([1 for _ in range(degree_of_fit+1)])
model_fit = new_model.fit(x, y, deg=degree_of_fit)
x_fit, y_fit = model_fit.linspace(n=100, domain=[-5, 5])

mean_squared_error(y, y_fit)

In [ ]:
plt.scatter(x, y, alpha=0.4, label='True')
plt.scatter(x_fit, y_fit, label='Fit')

plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.title(f"Fit y={model_fit}")
plt.show()

Wow! Loss here is a lot better. 

We have a different problem here though, one's that a bit more subtle. It's easier to see this when we check the truth value of the source polynominal against our predictions. 

In [ ]:
_, y_true = polynominal.linspace(n=100, domain=[-5, 5])
x, y_pred = model_fit.linspace(n=100, domain=[-5, 5])

plt.scatter(x, y_true, label="True", alpha=0.6)
plt.scatter(x, y_pred, label="Prediction", alpha=0.6)

plt.title(f"Fit with MSE of {round(mean_squared_error(y_true, y_pred), 2)}")
plt.legend() 
plt.show() 

# If our model was truly good, it would be a nearly perfect fit when noise is removed. Instead, we fit to the noise. 

# Challenge: Write a perfect model

Here you only have one parameter, the degree of the fit. Can you perfectly match the given polynominal? You can either use guess and check or be clever about it! 

🤖 Using np.Polynomal, fit data stored in the x, y variables. Assume x and y already exists. The degree of the polynominal is unknown. You can check the result with the true polynomial using '_, y_true = polynominal.linspace(...)'. Help me write a method that will help me pick the best degree of the polynominal.

In [ ]:
degree_of_fit = "?"

new_model = poly.polynomial.Polynomial([1 for _ in range(degree_of_fit+1)])
new_model = new_model.fit(x, y, deg=degree_of_fit)
x_fit, y_fit = new_model.linspace(n=100, domain=[-5, 5])

In [ ]:
_, y_true = polynominal.linspace(n=100, domain=[-5, 5])

plt.scatter(x, y_true, label="True", alpha=0.6)
plt.scatter(x, y_fit, label="Prediction", alpha=0.6)

plt.title(f"Fit with MSE of {round(mean_squared_error(y_true, y_fit), 2)}")
plt.legend() 
plt.show() 